In [16]:
import pandas as pd
import pdfplumber
import pandas as pd
import re
from tableauscraper import TableauScraper as TS


In [12]:
import pdfplumber
import pandas as pd
import re

PDF_PATH = "SHDGR_INV_8_J.pdf"   # <-- change this
START_PAGE = 27               # 1-based page number where the table begins
OUTPUT_CSV = "conseils_de_guerre.csv"

# Column bounding boxes (x0, x1) — tuned to the page layout.
# Adjust these if your PDF has different margins/column widths.
# Based on the sample image (page width ~595pt for A4):
COLUMNS = {
    "NOM":      (30,  175),
    "PRÉNOMS":  (175, 330),
    "CONSEIL":  (330, 390),
    "DOSSIER":  (390, 470),
    "GR 8 J":   (470, 570),
}

def extract_cell_text(words, x0, x1, y0, y1):
    """Collect words that fall within a bounding box."""
    cell_words = [
        w["text"] for w in words
        if w["x0"] >= x0 - 2
        and w["x1"] <= x1 + 2
        and w["top"] >= y0 - 2
        and w["bottom"] <= y1 + 2
    ]
    return " ".join(cell_words).strip()

def extract_table_from_page(page):
    """
    Extract rows from a single page using horizontal lines to detect row boundaries.
    Falls back to word-clustering if no lines are found.
    """
    rows = []
    words = page.extract_words(x_tolerance=3, y_tolerance=3)
    if not words:
        return rows

    # Get horizontal lines to identify row separators
    lines = page.lines or []
    h_lines = sorted(
        set(round(l["top"]) for l in lines if abs(l["top"] - l["bottom"]) < 2),
    )

    # If we have enough lines, use them as row boundaries
    if len(h_lines) > 5:
        boundaries = list(zip(h_lines, h_lines[1:]))
    else:
        # Fallback: cluster words by their vertical midpoint
        y_positions = sorted(set(round((w["top"] + w["bottom"]) / 2) for w in words))
        # Merge positions within 4pt of each other
        clusters = []
        for y in y_positions:
            if clusters and y - clusters[-1] < 6:
                clusters[-1] = (clusters[-1] + y) / 2
            else:
                clusters.append(y)
        # Build boundaries: each cluster ± half the typical row height
        row_h = 12  # approximate row height in points
        boundaries = [(c - row_h / 2, c + row_h / 2) for c in clusters]

    for y0, y1 in boundaries:
        row = {col: extract_cell_text(words, cx0, cx1, y0, y1)
               for col, (cx0, cx1) in COLUMNS.items()}
        # Skip empty rows and the header row
        if not any(row.values()):
            continue
        if row["NOM"].upper() in ("NOM", ""):
            continue
        rows.append(row)

    return rows


def merge_continuation_rows(rows):
    """
    Some entries span two lines (e.g. GARDEMBOIS with two DOSSIER numbers).
    Merge a row into the previous one if NOM and PRÉNOMS are both empty.
    """
    merged = []
    for row in rows:
        if not row["NOM"] and not row["PRÉNOMS"] and merged:
            prev = merged[-1]
            for col in ("CONSEIL", "DOSSIER", "GR 8 J"):
                if row[col]:
                    prev[col] = prev[col] + " / " + row[col] if prev[col] else row[col]
        else:
            merged.append(row)
    return merged


all_rows = []

with pdfplumber.open(PDF_PATH) as pdf:
    total_pages = len(pdf.pages)
    print(f"PDF has {total_pages} pages. Extracting from page {START_PAGE} onwards...")

    for page_num in range(START_PAGE - 1, total_pages):  # 0-based index
        page = pdf.pages[page_num]
        rows = extract_table_from_page(page)
        rows = merge_continuation_rows(rows)
        all_rows.extend(rows)

        if (page_num - START_PAGE + 2) % 50 == 0:
            print(f"  Processed page {page_num + 1} / {total_pages} "
                  f"({len(all_rows)} rows so far)")

df = pd.DataFrame(all_rows, columns=list(COLUMNS.keys()))

# Clean up: strip whitespace, normalize multiple spaces
for col in df.columns:
    df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

# Remove any remaining pure-header rows that slipped through
df = df[~df["NOM"].str.upper().isin(["NOM", "CONSEILS DE GUERRE DE LA COMMUNE DE PARIS 1871"])]
df = df[df["NOM"].str.len() > 0]  # drop fully empty NOM rows

df.reset_index(drop=True, inplace=True)

print(f"\nExtraction complete: {len(df)} rows extracted.")
print(df.head(10).to_string())

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved to {OUTPUT_CSV}")


PDF has 518 pages. Extracting from page 27 onwards...
  Processed page 76 / 518 (1579 rows so far)
  Processed page 126 / 518 (3130 rows so far)
  Processed page 176 / 518 (4653 rows so far)
  Processed page 226 / 518 (6178 rows so far)
  Processed page 276 / 518 (7821 rows so far)
  Processed page 326 / 518 (9379 rows so far)
  Processed page 376 / 518 (10888 rows so far)
  Processed page 426 / 518 (12396 rows so far)
  Processed page 476 / 518 (14170 rows so far)

Extraction complete: 15390 rows extracted.
       NOM           PRÉNOMS CONSEIL DOSSIER   GR 8 J
0      AAB    Pierre, Eugène               3   168 12
1   ABADIE           Antoine              11  371 311
2   ABADIE  Charles, Antoine               6  594 226
3   ABADIE     Ismaël, Isaac               4  705 132
4    ABART         Hippolyte              23  243 451
5    ABART             Louis              10  175 292
6     ABEL            Pierre              14  347 353
7  ABERLEN          François              14  457 356


In [13]:
df = pd.read_csv('conseils_de_guerre.csv')
df.head()

,NOM,PRÉNOMS,CONSEIL,DOSSIER,GR 8 J
0,AAB,"Pierre, Eugène",NaN,3,168 12
1,ABADIE,Antoine,NaN,11,371 311
2,ABADIE,"Charles, Antoine",NaN,6,594 226
3,ABADIE,"Ismaël, Isaac",NaN,4,705 132
4,ABART,Hippolyte,NaN,23,243 451


In [14]:
# append any non-empty CONSEIL text to PRÉNOMS (with a space), then clear CONSEIL
mask = df['CONSEIL'].notna() & df['CONSEIL'].str.strip().ne('')
df.loc[mask, 'PRÉNOMS'] = df.loc[mask, 'PRÉNOMS'].fillna('').astype(str).str.strip() + ' ' + df.loc[mask, 'CONSEIL'].str.strip()
df['PRÉNOMS'] = df['PRÉNOMS'].str.strip()
df.loc[mask, 'CONSEIL'] = ''

# quick check
df.head()

,NOM,PRÉNOMS,CONSEIL,DOSSIER,GR 8 J
0,AAB,"Pierre, Eugène",NaN,3,168 12
1,ABADIE,Antoine,NaN,11,371 311
2,ABADIE,"Charles, Antoine",NaN,6,594 226
3,ABADIE,"Ismaël, Isaac",NaN,4,705 132
4,ABART,Hippolyte,NaN,23,243 451


In [15]:
# move existing DOSSIER text into CONSEIL, then repopulate DOSSIER from the first token of "GR 8 J"
df['CONSEIL'] = df['CONSEIL'].fillna('').astype(str)
df['DOSSIER'] = df['DOSSIER'].fillna('').astype(str)

# append old DOSSIER into CONSEIL (use " / " as separator), then clear DOSSIER
mask = df['DOSSIER'].str.strip() != ''
df.loc[mask, 'CONSEIL'] = (df.loc[mask, 'CONSEIL'].str.strip() + ' / ' + df.loc[mask, 'DOSSIER'].str.strip()).str.strip(' / ')
df['DOSSIER'] = ''

# split "GR 8 J" on the first whitespace: first token -> DOSSIER, remainder -> GR 8 J
gr = df['GR 8 J'].fillna('').astype(str).str.strip()
parts = gr.str.split(r'\s+', n=1, expand=True)
df['DOSSIER'] = parts[0].fillna('').astype(str).str.strip()
df['GR 8 J'] = parts[1].fillna('').astype(str).str.strip()

# normalize whitespace
for col in ['CONSEIL', 'DOSSIER', 'GR 8 J']:
    df[col] = df[col].astype(str).str.strip().replace(r'\s+', ' ', regex=True)

# save results
df.to_csv(csv_outputname, index=False, encoding='utf-8-sig')

# show a quick preview
df.head()

,NOM,PRÉNOMS,CONSEIL,DOSSIER,GR 8 J
0,AAB,"Pierre, Eugène",3,168,12
1,ABADIE,Antoine,11,371,311
2,ABADIE,"Charles, Antoine",6,594,226
3,ABADIE,"Ismaël, Isaac",4,705,132
4,ABART,Hippolyte,23,243,451


In [5]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import string
import re

def scrape_communards_refined():
    base_url = "https://communards-1871.fr/index.php"
    all_data = []
    
    for letter in string.ascii_uppercase:
        print(f"Scraping letter: {letter}...")
        params = {'page': 'recherches/liste_nominative', 'filtre': letter}
        
        try:
            response = requests.get(base_url, params=params, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            table = soup.find('table')
            if not table: continue
                
            for row in table.find_all('tr')[1:]:
                cols = row.find_all('td')
                if len(cols) < 3: continue # Ensure we have enough columns
                
                # --- FIX: Column 1 (Surname/Name + URL) ---
                link_tag = cols[0].find('a')
                name = link_tag.text.strip() if link_tag else cols[0].text.strip()
                profile_url = f"https://communards-1871.fr/{link_tag['href']}" if link_tag and 'href' in link_tag.attrs else None
                
                # --- Column 2: First Name ---
                # Based on your previous data structure, first names are often in the second column
                first_name = cols[1].text.strip()
                
                # --- Column 3 & 4: Age and Occupation ---
                # We assume the third column is age and fourth is occupation based on your request
                # If the table layout differs, you might need to adjust indices
                age = cols[2].text.strip()
                occupation = cols[3].text.strip()
                
                all_data.append({
                    "Surname": name,
                    "First_Name": first_name,
                    "Age": age,
                    "Occupation": occupation,
                    "Profile_URL": profile_url
                })
                
            time.sleep(2)
        except Exception as e:
            print(f"Error on {letter}: {e}")
            
    return pd.DataFrame(all_data)

df = scrape_communards_refined()
df.to_csv("communards_structured.csv", index=False, encoding='utf-8-sig')
print("Successfully extracted structured data with embedded URLs.")

Scraping letter: A...
Scraping letter: B...
Scraping letter: C...
Scraping letter: D...
Scraping letter: E...
Scraping letter: F...
Scraping letter: G...
Scraping letter: H...
Scraping letter: I...
Scraping letter: J...
Scraping letter: K...
Scraping letter: L...
Scraping letter: M...
Scraping letter: N...
Scraping letter: O...
Scraping letter: P...
Scraping letter: Q...
Scraping letter: R...
Scraping letter: S...
Scraping letter: T...
Scraping letter: U...
Scraping letter: V...
Scraping letter: W...
Scraping letter: X...
Scraping letter: Y...
Scraping letter: Z...
Successfully extracted structured data with embedded URLs.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_communards_detailed(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        container = soup.find(id='impression')
        if not container:
            return None
        
        # 1. Capture Raw Text Buffer for Archival Integrity
        raw_text = container.get_text(separator=' ', strip=True)
        
        # 2. Prepare Dictionary
        data = {"Profile_URL": url, "Raw_Bio_Text": raw_text}
        
        # 3. Locate all Bold Labels
        bold_elements = container.find_all('b')
        
        for i, b_tag in enumerate(bold_elements):
            label = b_tag.get_text(strip=True).replace(":", "").strip()
            
            # Stop condition: If label indicates start of archival files
            if "Dossiers individuels" in label:
                break
                
            # 4. Extract Value: Everything between this <b> and the next <b>
            # We look at the content following the <b> tag
            value_parts = []
            current = b_tag.next_sibling
            
            # Collect text until we hit the next <b> tag or end of container
            while current and current.name != 'b':
                if hasattr(current, 'text'):
                    value_parts.append(current.get_text(strip=True))
                elif isinstance(current, str):
                    value_parts.append(current.strip())
                current = current.next_sibling
            
            value = " ".join([v for v in value_parts if v]).strip()
            if value.startswith(":"):
                value = value[1:].strip()
                
            data[label] = value
            
        return data
        
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

# --- Usage Strategy ---
# 1. Start with your list of URLs from the previous nominative scraper
# 2. Iterate through them:
# all_profiles = [scrape_communards_detailed(url) for url in list_of_urls]
# df = pd.DataFrame(all_profiles)
# df.to_csv("communards_full_profiles.csv", index=False)

In [ ]:
import pandas as pd
import os

# Create an empty list or start with an existing file if you are restarting
results = []
checkpoint_file = "communards_full_dataset.csv"

# Start from where you left off if the file already exists
# (This is advanced: you'd check against your master URL list)
urls = df['Profile_URL'].tolist()

for i, url in enumerate(urls):
    # 1. Scrape the record
    data = scrape_communards_detailed(url)
    if data:
        results.append(data)
    
    # 2. Checkpoint: Save every 50 records
    if (i + 1) % 50 == 0:
        temp_df = pd.DataFrame(results)
        temp_df.to_csv(checkpoint_file, index=False)
        print(f"Progress: {i + 1}/{len(urls)} saved.")

# Final save
pd.DataFrame(results).to_csv(checkpoint_file, index=False)

Progress: 50/41375 saved.
Progress: 100/41375 saved.
Progress: 150/41375 saved.
Progress: 200/41375 saved.
Progress: 250/41375 saved.
Progress: 300/41375 saved.
Progress: 350/41375 saved.
Progress: 400/41375 saved.
Progress: 450/41375 saved.
Progress: 500/41375 saved.
Progress: 550/41375 saved.
Progress: 600/41375 saved.
Progress: 650/41375 saved.
Progress: 700/41375 saved.
Progress: 750/41375 saved.
Progress: 800/41375 saved.
Progress: 850/41375 saved.
Progress: 900/41375 saved.
Progress: 950/41375 saved.
Progress: 1000/41375 saved.
Progress: 1050/41375 saved.
Progress: 1100/41375 saved.
Progress: 1150/41375 saved.
Progress: 1200/41375 saved.
Progress: 1250/41375 saved.
Progress: 1300/41375 saved.
Progress: 1350/41375 saved.
Progress: 1400/41375 saved.
Progress: 1450/41375 saved.
Progress: 1500/41375 saved.
Progress: 1550/41375 saved.
Progress: 1600/41375 saved.
Progress: 1650/41375 saved.
Progress: 1700/41375 saved.
Progress: 1750/41375 saved.
Progress: 1800/41375 saved.
Progress: 18